# open the tiff data

to get familiarized with it

There are some issues with the following files (Virtualizarr cannot open them)

- input_files/NSIDC-0477_AMSR_37V_CO_FT_2023_day365_v05.2.tif




In [ ]:
from pathlib import Path
import json
import xarray as xr

## virtualizarr specific libraries
from virtualizarr import open_virtual_dataset
from virtual_tiff import VirtualTIFF
from obstore.store import LocalStore
from obspec_utils.registry import ObjectStoreRegistry

# pydap specific libraries
from pydap.model import DatasetType
from pydap.responses.dmr import DMRResponse
from pydap.parsers.dmr import DummyData

## Kerchunk files

## Input tif files

We work directly with the tif file.



In [ ]:
directory_path = Path("./input_files/")

# Find all .json files in the top-level directory
tif_files = list(directory_path.glob("*.tif"))
print("found: ", len(tif_files), " tif files")
filename = tif_files[0]
filename

## Virtual datastore

The end result is an Xarray Dataset object. Will use an intermediate product, to translate the information into the pydap data model


In [ ]:
filepath = f"{filename.resolve().parent}/{filename.name}"

registry = ObjectStoreRegistry({"file://": LocalStore()})
parser = VirtualTIFF(ifd_layout="nested")

ms = parser(f"file://{filepath}", registry=registry)
ds = ms.to_virtual_datatree()
ds

## Parsing metadata

The following steps will enable the creation of a pydap dataset, and with it, the generation of a dmrpp



In [ ]:
groups = list(ms._group.groups.keys())
groups

In [ ]:
ms._group.arrays # root arrays?

In [ ]:
ms._group.metadata.attributes

## Hierarchical data

Extract any nested data and place it within Groups


In [ ]:
GROUPS = {}
for group in groups:
    arrays={}
    for k,v in ms._group[group].arrays.items():
        chunk_manifest = {"chunk_shape": v.metadata.chunk_grid.chunk_shape, "fill_value": v.metadata.fill_value, "codecs": v.metadata.codecs,"hrefs": v.manifest.dict()}
        arrays.update({k:{'shape': v.shape, 'dtype': v.dtype, "dims": v.metadata.dimension_names, "chunk_manifest": chunk_manifest}})
    GROUPS.update({group:{"attributes": ms._group[group].metadata.attributes, "arrays": arrays}})

The nested dictionary holds information about the array data that will go into the dmrpp

In [ ]:
GROUPS['0']['arrays']['0'].keys()

In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest'].keys()

In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest']['codecs']

In [ ]:
ms._group['0'].arrays['0']

In [ ]:
ms._group['0'].arrays['0'].metadata.codecs

## Populate the pydap dataset

Here, the virtualizarr metadata is injected into the pydap data model


In [ ]:
def array_metadata(arraymeta: dict, parent: str | None = None):
    """Reads array medatata extracteed from virtualizarr, and
    and re structures it following pydap-specific syntax
    """
    if not parent:
        parent = "/"
    _dims_shapes = dict((dim, size) for dim, size in zip(arraymeta['dims'], arraymeta['shape']))
    _dims = ["/".join([parent,dim]) for dim in list(_dims_shapes)]
    _data = DummyData(dtype= arraymeta['dtype'], shape= arraymeta['shape'], path = parent)
    return _dims, _dims_shapes, _data

In [ ]:
pyds = DatasetType(name=filename.name, attributes=ms._group.metadata.attributes)
for array in ms._group.arrays:
    dims = ms._group[array].metadata.dimensions_names
    data = DummyData(dtype=ms._group[array].dtype, shape=ms._group[array].shape, path='/')
    pyds.createVariable(name=array, dims=dims, data=data)

### Now all hierarchical data
for gr in GROUPS:
    _DIMS = {}
    group_name = "/" + gr
    pyds.createGroup(name=group_name, attributes=GROUPS[gr]['attributes'])
    for array in GROUPS[gr]["arrays"]:
        var_name = "/".join([group_name,array])
        dims, dim_shapes, data = array_metadata(arraymeta = GROUPS[gr]["arrays"][array], parent=group_name)
        pyds.createVariable(name = var_name, dims = dims, data=data)
        _DIMS.update(dim_shapes)
    pyds[group_name].attributes['dimensions'] = _DIMS

In [ ]:
dmr_name = f"./output_files/{filename.name}.dmr"
dmr_name

In [ ]:
with open(dmr_name, "wb") as file:
    file.write(b"".join(DMRResponse(pyds)).decode("ascii").encode("utf-8"))

# What is missing?

All the `++` elements of the dmrpp. All the information has been extracted in the `GROUPS` nested dictionary already. For example, the missing info is in:

```python
gr = "<group_name_here>"
array = "<array_name_here>"
GROUPS[gr]["arrays"][array]["chunk_manifest"].keys()
>>> dict_keys(['chunk_shape', 'fill_value', 'codecs', 'hrefs'])
```



## What is needed

1. Translate the zarr chunk syntax into the dmrpp chunk synthax. For example in the case the `chunk_shape = (512,512)`:

* Chunks in zarr -> {'0.0', '0.1', ...}

* chunks in dmrpp -> {'[512, 512]', '[512, 1024]'}

* 0.0 -> [256,256]

* 0.1 -> [512, 1024]

2. Once that is translated, this informations needs to be added into the pydap dataset, per array. The easiest way is to expand on the `DummyData` class defined in `pydap.parsers.dmr`. We need to add all the (basic) elements of the dmrpp into this class. Currently the DummyData class only takes `dtype`, `shape`, and `path` (which is the parent). Every new element must be optional so it does not break any existing workflow.

3. Once the `DummyClass` has been expanded to include all (optional) dmrpp elements, the `DMRResponse` in `pydap.responses.dmr` needs to be expanded to extract this (optional) information per array. This is should not be too hard, since almost all machinery is there already.




In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest'].keys()

In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest']['chunk_shape']

In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest']['fill_value']

In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest']['codecs']

In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest']['hrefs']

## Translating `chunk_manifest` into `dmrpp:chunks` XML

Translates the zarr-style chunk manifest (`chunk_shape`, `fill_value`, `codecs`, `hrefs`) extracted per array into the `<dmrpp:chunks>` / `<dmrpp:chunk>` XML fragment used inside a dmrpp file.

Zarr chunk keys such as `"1.2"` are converted into DAP's `chunkPositionInArray`, e.g. for `chunk_shape=(256,256)`, key `"1.2"` -> `[256, 512]` (index * chunk_shape per dimension).

In [ ]:
def _chunk_key_to_position(key: str, chunk_shape: tuple) -> list:
    """Translate a zarr chunk key (e.g. '1.2') into the dmrpp
    `chunkPositionInArray` (e.g. [256, 512] for chunk_shape=(256,256)).
    """
    indices = [int(i) for i in key.split(".")]
    return [idx * size for idx, size in zip(indices, chunk_shape)]


# maps zarr/numcodecs/imagecodecs compressor names onto dmrpp compressionType values
_COMPRESSOR_TO_DMRPP = {
    "numcodecs.zlib": "deflate",
    "numcodecs.gzip": "deflate",
    "imagecodecs_lzw": "lzw",
}


def _codec_attributes(codecs) -> dict:
    """Inspect a zarr/virtual-tiff codecs pipeline and pull out the bits that
    map onto `dmrpp:chunks` attributes: byteOrder, compressionType and
    deflateLevel. Filter/predictor codecs (delta, transpose, float-predictor,
    etc.) don't have a dmrpp equivalent and are ignored.
    """
    attrs = {"byteOrder": "LE"}
    for codec in codecs:
        endian = getattr(codec, "endian", None)
        if endian is not None:
            attrs["byteOrder"] = "BE" if "big" in str(endian).lower() else "LE"
        codec_name = getattr(codec, "codec_name", None)
        if codec_name in _COMPRESSOR_TO_DMRPP:
            attrs["compressionType"] = _COMPRESSOR_TO_DMRPP[codec_name]
            level = getattr(codec, "codec_config", {}).get("level")
            if level is not None:
                attrs["deflateLevel"] = level
    return attrs


def chunk_manifest_to_dmrpp_chunks(chunk_manifest: dict, indent: str = "") -> str:
    """Translate a `chunk_manifest` dict (as stored per array in
    `GROUPS[group]["arrays"][array]["chunk_manifest"]`) into the
    `<dmrpp:chunks>...</dmrpp:chunks>` XML fragment used inside a dmrpp file.

    Parameters
    ----------
    chunk_manifest : dict
        Must contain 'chunk_shape', 'fill_value', 'codecs' and 'hrefs' keys.
    indent : str
        Base indentation prefixed to every emitted line.

    Returns
    -------
    str
        The `<dmrpp:chunks>` XML fragment, ready to be inserted inside the
        corresponding DAP variable element of a dmrpp file.
    """
    chunk_shape = chunk_manifest["chunk_shape"]
    fill_value = chunk_manifest["fill_value"]
    hrefs = chunk_manifest["hrefs"]

    codec_attrs = _codec_attributes(chunk_manifest["codecs"])

    attrs = {}
    if "compressionType" in codec_attrs:
        attrs["compressionType"] = codec_attrs["compressionType"]
    if "deflateLevel" in codec_attrs:
        attrs["deflateLevel"] = codec_attrs["deflateLevel"]
    if fill_value is not None:
        attrs["fillValue"] = fill_value
    attrs["byteOrder"] = codec_attrs["byteOrder"]
    attrs_str = " ".join(f'{name}="{value}"' for name, value in attrs.items())

    lines = [f"{indent}<dmrpp:chunks {attrs_str}>"]
    lines.append(
        f"{indent}    <dmrpp:chunkDimensionSizes>"
        f"{' '.join(str(s) for s in chunk_shape)}</dmrpp:chunkDimensionSizes>"
    )
    for key in sorted(hrefs, key=lambda k: [int(i) for i in k.split(".")]):
        chunk = hrefs[key]
        position = "[" + ",".join(str(p) for p in _chunk_key_to_position(key, chunk_shape)) + "]"
        lines.append(
            f'{indent}    <dmrpp:chunk offset="{chunk["offset"]}" '
            f'nBytes="{chunk["length"]}" chunkPositionInArray="{position}" '
            f'href="{chunk["path"]}"/>'
        )
    lines.append(f"{indent}</dmrpp:chunks>")
    return "\n".join(lines)

In [ ]:
print(chunk_manifest_to_dmrpp_chunks(GROUPS['0']['arrays']['0']['chunk_manifest'], indent="        "))

## Writing the `.dmrpp` file

Insert each array's `<dmrpp:chunks>` element into a copy of the generated `.dmr` file, matching by group name and array name, and write the result next to the source `.dmr` with a `.dmrpp` extension.

In [ ]:
import xml.etree.ElementTree as ET

DAP4_NS = "http://xml.opendap.org/ns/DAP/4.0#"
DMRPP_NS = "http://xml.opendap.org/dap/dmrpp/1.0.0#"
ET.register_namespace("", DAP4_NS)
ET.register_namespace("dmrpp", DMRPP_NS)


def _build_dmrpp_chunks_element(chunk_manifest: dict) -> ET.Element:
    """Build the `<dmrpp:chunks>` element (with its `<dmrpp:chunk>` children)
    for one array's chunk_manifest, ready to be appended to that array's
    variable element in a parsed DMR tree.
    """
    chunk_shape = chunk_manifest["chunk_shape"]
    fill_value = chunk_manifest["fill_value"]
    hrefs = chunk_manifest["hrefs"]

    codec_attrs = _codec_attributes(chunk_manifest["codecs"])

    attrib = {}
    if "compressionType" in codec_attrs:
        attrib["compressionType"] = codec_attrs["compressionType"]
    if "deflateLevel" in codec_attrs:
        attrib["deflateLevel"] = str(codec_attrs["deflateLevel"])
    if fill_value is not None:
        attrib["fillValue"] = str(fill_value)
    attrib["byteOrder"] = codec_attrs["byteOrder"]

    chunks_el = ET.Element(f"{{{DMRPP_NS}}}chunks", attrib)
    dim_sizes_el = ET.SubElement(chunks_el, f"{{{DMRPP_NS}}}chunkDimensionSizes")
    dim_sizes_el.text = " ".join(str(s) for s in chunk_shape)

    for key in sorted(hrefs, key=lambda k: [int(i) for i in k.split(".")]):
        chunk = hrefs[key]
        position = "[" + ",".join(str(p) for p in _chunk_key_to_position(key, chunk_shape)) + "]"
        ET.SubElement(chunks_el, f"{{{DMRPP_NS}}}chunk", {
            "offset": str(chunk["offset"]),
            "nBytes": str(chunk["length"]),
            "chunkPositionInArray": position,
            "href": chunk["path"],
        })
    return chunks_el


def _find_variable_element(root: ET.Element, group_name: str, array_name: str) -> ET.Element:
    """Locate the DAP4 variable element for `array_name` inside the
    top-level Group named `group_name` (e.g. group_name="0", array_name="0").
    """
    group_el = root.find(f"{{{DAP4_NS}}}Group[@name='{group_name}']")
    if group_el is None:
        raise ValueError(f"Group '{group_name}' not found in DMR")
    for child in group_el:
        if child.tag == f"{{{DAP4_NS}}}Dimension":
            continue
        if child.get("name") == array_name:
            return child
    raise ValueError(f"Variable '{array_name}' not found in group '{group_name}'")


def add_chunk_manifests_to_dmr(dmr_path, groups: dict, output_path=None) -> Path:
    """Read a `.dmr` file, insert each array's `dmrpp:chunks` element (built
    from `groups[group]["arrays"][array]["chunk_manifest"]`) into its
    matching variable, and write the result to a `.dmrpp` file next to the
    source `.dmr`.

    Parameters
    ----------
    dmr_path : str | Path
        Path to the source `.dmr` file (as written by `DMRResponse`).
    groups : dict
        The `GROUPS` dictionary, i.e.
        `{group_name: {"arrays": {array_name: {"chunk_manifest": {...}}}}}`.
    output_path : str | Path, optional
        Defaults to `dmr_path` with its suffix replaced by `.dmrpp`.

    Returns
    -------
    Path to the written `.dmrpp` file.
    """
    dmr_path = Path(dmr_path)
    tree = ET.parse(dmr_path)
    root = tree.getroot()

    for group_name, group_data in groups.items():
        for array_name, array_data in group_data["arrays"].items():
            var_el = _find_variable_element(root, group_name, array_name)
            var_el.append(_build_dmrpp_chunks_element(array_data["chunk_manifest"]))

    ET.indent(tree, space="    ")
    output_path = Path(output_path) if output_path else dmr_path.with_suffix(".dmrpp")
    tree.write(output_path, encoding="ISO-8859-1", xml_declaration=True)
    return output_path

In [ ]:
dmrpp_name = add_chunk_manifests_to_dmr(dmr_name, GROUPS)
dmrpp_name

In [ ]:
print(dmrpp_name.read_text())